# Домашнее задание: Сбор данных и разметка: от формулировки задачи до крауда

В этом кейсе вы пройдёте путь **от постановки бизнеса** до **пайплайна гибридной разметки**:  
Постановка задачи → разметка zero shot промптом с помощью LLM → оценка качества → Улучшение качества промпта: few-shot, cot и другие способы → Оценка уверенности ответа




### Установка зависимостей

In [ ]:
import torch, json, random, re, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)


Device: cuda


In [ ]:
# !pip uninstall -y fsspec datasets
# !pip install fsspec==2024.2.0 datasets==2.18.0

## 1. Постановка задачи


**Контекст (от лица бизнеса):**
Наша компания разрабатывает финтех-приложение с поддержкой пользователей через чат-бот.
Мы хотим автоматически определять тему запроса клиента (например: "блокировка карты", "потеря ПИН-кода", "перевыпуск карты", и т.д.), чтобы быстро направлять клиента к нужному решению. Требуется получить данные для задачи




**Описание задачи:**
> "Для каждого входящего текстового сообщения пользователя автоматически определить одну из тематик (например, balance, card_not_working, transfer, etc.)"

## 2. Требования и бизнес-метрики – 1 балл

Предложите не менее 2 бизнес-метрик, которые может хотеть оптимизировать бизнес относительно процесса разметки данных для данной задачи.


In [ ]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
Хорошими метриками будет X, Y и Z, потому что так и есть
""")
# ---- Конец кода ----

## 3. Сведение к ML-задаче – 2 балла



Сведите бизнес-задачу к задаче машинного обучения, опишите входные данные и метки:

- **Тип задачи**:
- **Объект**:
- **Метки**:

In [ ]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
- **Тип задачи**:
- **Объект**:
- **Метки**:
""")
# ---- Конец кода ----

## 4. ML-метрики – 2 балла


Сформулируйте, какие метрики вы будете отслеживать в процессе сбора данных и получения разметки: как при помощи LLM, так и при помощи разметчиков в крауде

In [ ]:
# ---- Ваш код здесь ----
print("""
- LLM разметчик: (метрики)
- Разметчики в крауде: (метрики)

""")
# ---- Конец кода ----



## 5. Данные и бейзлайн разметка

### 5.1 Загрузка и первичный анализ датасета

Посмотрим данные: примеры из датасета (попробуем разметить сами хотя бы 10 примеров), все типы меток, размеры выборок, распределение

In [ ]:
# !pip uninstall -y fsspec datasets
# !pip install fsspec==2024.2.0 datasets==2.18.0

from datasets import load_dataset
import pandas as pd

ds = load_dataset('banking77')


# ---- Ваш код здесь ----
print("""
Считываем данные
""")
# ---- Конец кода ----

### 5.2 Бейзлайн LLM разметка (7 баллов)

В этом пункте нужно получить бейзлайн разметку с помощью open source LLM и простого короткого промпта.

Для упрощения тут у нас уже есть golden set разметка (в случае если не было бы, то действовали как указано в лекции, или бы размечали для начала сами хотя бы 50-100 примеров), на которой мы можем проверять качество

В качестве LLM можно также взять https://huggingface.co/IlyaGusev/saiga_yandexgpt_8b . Либо можно заменить на любую open-source (например "Qwen/Qwen3-8B", "meta-llama/Meta-Llama-3-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.2", и т.д.)

In [ ]:
# !pip install -U accelerate bitsandbytes transformers

In [ ]:
# ---- Ваш код здесь ----
print("""
    Загружаем выбранную LLM
    В некоторых случаях может понадобиться !pip install -U accelerate bitsandbytes transformers
""")
# ---- Конец кода ----

Формируем простой короткий промпт, в котором укажем  все категории меток для разметки, и потребуем выход в нужном формате json {"label": str}, для которого напишем функцию парсинга



In [ ]:
# ---- Ваш код здесь ----
prompt_template = (
    "Текст промпта для разметки"
    )
# ---- Конец кода ----


Делаем разметку 10-20 примеров, пишем функцию парсинга ответа (считаем метрику в скольких ответах нарушения следования формату), смотрим ответы

In [1]:
# Функция разметки вместе с промптом,

# ---- Ваш код здесь ----
print("""
    прокачиваем в цикле выбранную LLM для разметки данных через функцию annotate, добавляем разметку в исходный датасет и сохраняем в файл
""")
# ---- Конец кода ----



    прокачиваем в цикле выбранную LLM для разметки данных через функцию annotate, добавляем разметку в исходный датасет и сохраняем в файл



### 5.3 Оценка качества (2 балла)

Оцениваем качество разметки на тестовом датасет (либо на семпле из тестового датасета)



In [ ]:
# ---- Ваш код здесь ----
print("""
    Инферим LLM на тесте, замеряем метрики
""")
# ---- Конец кода ----



### 6. Улучшение качества промпта

### 6.1 few shot prompt (4 баллов)

Добавим в промпт few-shot примеры (важно, чтобы не было data leak  c тестом): помогает исправлять поведение модели, когда описание в инструкции не справляется + модель лучше следует форматам

In [ ]:
# ---- Ваш код здесь ----
print("""
    Пишем промпт с few shot, замеряем качество
""")
# ---- Конец кода ----



### 6.2 chain-of-thoughts (3 балла)

Добавляем сhain-of-thought в промпт: требуем короткий reasoning → повышаем прозрачность решения модели (понимаем, почему размечено именно так) и улучшаем качество на более сложных задачах

Json на выходе теперь формата
{"reasoning": "why_this_class", "label": "one_of_the_categories"}

In [ ]:
# ---- Ваш код здесь ----
print("""
    Пишем промпт с few shot, замеряем качество
""")
# ---- Конец кода ----



### 6.3 Дальнейшие улучшения (6 баллов)

Далее улучшаем итеративно

Основные улучшения в общем случае происходит за счет:
- Аналитика ошибок,  в первую очередь анализируем ошибки разметки (только на трейне, чтобы не подогнаться под тест!), в том числе используя reasoning модели, чтобы понять причины. Также помогает спрашивать у самой модели и просить ее поправить начальный промпт/инструкцию

- Понимание бизнеса и домена, четкое описание в инструкции/промпте

Дополнительно, что тут может еще помочь:
- Упрощение задачи: размечать не одну, а несколько наиболее релеватных меток для каждого текста (=> растим recall)

- Использовать более "умные" LLM
- Размечаем с перекрытием: запускаем промпт n раз (например, 3) и агрегируем ответ. Улучшение: агрегурем результат ансамбля разных LLM (среди тех же размеров например: Qwen-3 8b) , можно с тем же промптом, либо промпты могут отлчичаться между собой few shot примерами

Тут нужно реализовать одно из улучшений (из лекции: слайды 39-41, 46, либо списка выше, например, с перекрытием). Цель - улучшить последний результат по accuracy и побить baseline: accuracy = 0.6 на тестовой выборке

In [7]:
# ---- Ваш код здесь ----
print("""
    Промпты с улучшением качества (baseline: accuracy = 0.6), сравнение метрик, как каждое улучшение повляило
""")
# ---- Конец кода ----


    Промпты с улучшением качества (baseline: accuracy = 0.6), сравнение метрик, как каждое улучшение повляило



## 7.1. Уверенность ответа. (11 баллов)

Посчитаем уверенность модели ответов по модели. Полезно для гибкой схемы разметки, когда более сложные примеры отправляются на разметку асессорам, либо на модель побольше, или на доп разметку с доп перекрытием.

Рабочий бейзлайн  - усредение logprob для сгенерированного текста («confidence can be approximated with the average token-level probability of the generated span»)

Здесь нужно написать функцию для взятия уверенности модели, как указано ниже, и далее показать, что числовая «уверенность» (confidence), посчитанная по лог-вероятностям токенов, действительно коррелирует с тем, ошиблась модель или нет. Ожидается получение AUC>=0.6


https://cookbook.openai.com/examples/using_logprobs


In [6]:

# ---- Ваш код здесь ----
def annotate_conf(text: str,
                  max_new_tokens: int = 32
                 ) -> tuple[str, float, int, str]:
    """
    Размечает один запрос при помощи LLM b сразу возвращает числовую
    «уверенность» предсказания на основе лог-вероятностей сгенерированных
    токенов.

    ▸ Логика шага
      1. Формируем prompt (например с few-shot).
      2. Вызываем `model.generate(..., output_scores=True)` — получаем
         логиты (score-векторы) для каждого сгенерированного токена.
      3. Берём JSON-фрагмент ответа; если label не распарсился
         или не входит в `label_names`, ставим `corrupted = 1`.
      4. **Confidence** = `exp(mean(log p))` по токенам JSON-строки
         (кумулятивная вероятность: чем ближе к 1 — тем модель
         «увереннее» в своём выборе).

    Параметры
    ----------
    text : str
        Пользовательский запрос, который нужно классифицировать.
    max_new_tokens : int, optional
        Сколько токенов максимум разрешаем модели сгенерировать
        (включая JSON и возможный «хвост»).  Дефолт — 32.

    Возвращает
    ----------
    label : str
        Предсказанная категория (`label_names` или "other").
    confidence : float
        Уверенность модели (0 – 1).  Расчёт: exp(среднего log p токенов JSON).
    corrupted : int
        0 — JSON корректный и label ∈ `label_names`;
        1 — формат сломан **или** label не из списка.
    raw_generation : str
        Полный ответ LLM (полезно для дебага/визуализации).
    """
    return
# ---- Конец кода ----



In [ ]:
# ---- Ваш код здесь ----
print("""
1. Посчитайте roc_auc_score(y_true_bin, y_score), где
y_true_bin = 1, если модель угадала (`pred_label == true_label`)), иначе 0, y_score = confidence, который вернула annotate_conf
2. Сделайте выводы по полезности confidence
""")
# ---- Конец кода ----


## 8.1. Human-in-the-loop с шумным разметчиком

В реальных задачах разметку часто делают не идеальные эксперты, а обычные асессоры — они тоже ошибаются. Для достижения хорошего качества с ними используется разметка с перкрытием по N accесорам, а затем агрегируется разметка (например majority vote). Таким образом получаем **gold-метку** и меру "уверенности" ассесоров - насколько они сходятся в решении (например 0.75 - из 4 ассессоров 3 выбрали итоговую метку).

Часть разметки можно переложить на LLM, если уметь оценивать его уверенность.

В этом задании мы смоделируем простую схему:

- есть **gold-метка** `true_label` (используем только для оценки качества),
- есть один **"человек"-разметчик** с ошибками (`human_label`),
- есть предсказания LLM: `pred_label` и `confidence`.

Мы хотим построить **гибридную систему**:

- по умолчанию используем метку человека (`human_label`);
- если `confidence >= threshold` — считаем, что LLM очень уверен и берём его метку (`pred_label`).

#### 8.2.1. Симуляция "шумного" разметчика (3 балла)

Реализуйте функцию, которая по gold-меткам `true_labels` возвращает `human_labels`:

- с вероятностью `1 - error_rate` берётся правильная метка,
- с вероятностью `error_rate` — случайная *другая* метка из `label_names`  
  (можно взять, например, `error_rate = 0.25`).

1. Реализуйте функцию `simulate_noisy_human(true_labels, label_names, error_rate=0.25, random_state=42)`.
2. Посчитайте accuracy такого разметчика относительно `true_labels`.

Замечание: в реальных задачах расхождения между разметчиками и ошибочно проставленные метки зачастую не случайны. В данном задании симулируем шум для упрощения.

In [2]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.2. Гибридная схема разметки (4 балла)

Реализуйте функцию, которая комбинирует разметку человека и LLM по порогу уверенности:

- по умолчанию использует `human_labels`,
- если `confidence[i] >= threshold` — вместо метки человека берёт `pred_labels[i]`,
- возвращает:
  - `overall_acc` — итоговую accuracy гибридной разметки (относительно `true_labels`),
  - `coverage` — долю объектов, где использовали LLM (от 0 до 1).

1. Реализуйте функцию  
   `simulate_hybrid(pred_labels, human_labels, true_labels, confidence, threshold)`.
2. Проверьте её на небольшом игрушечном примере

In [3]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.3. Подбор порога и анализ trade-off (3 балла)

1. Для `threshold` из диапазона `[0.0, 1.0]` с шагом 0.05:
   - посчитайте `overall_acc` и `coverage`;
   - выведите таблицу с колонками `threshold`, `accuracy`, `coverage`  
     (по желанию можно дополнительно построить график `accuracy` vs `coverage`).
2. Найдите значение `threshold`, при котором:
   - качество гибридной разметки **не ниже** заданного уровня (например, accuracy ≥ 0.8 *относительно gold*),
   - а `coverage` максимально возможен.

In [4]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.4. Сравнение схем разметки (2 балла)

1. Посчитайте и сравните:
   - accuracy только человека (`human_labels`),
   - accuracy только LLM (`pred_labels`),
   - лучшую точку гибрида.
2. Кратко (2–3 предложения) ответьте:
   - в чём преимущество гибридного подхода по сравнению с «чисто человек» и «чисто LLM»?
   - в каких кейсах такая схема human-in-the-loop может быть особенно полезна?


In [5]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


## Итоги домашки

В этой работе мы посмотрели на разметку как на систему, где есть и люди, и LLM.

Главное, что нужно вынести:
- LLM можно использовать как разметчика (при этом важно следить за качеством ), можно улучшать промт за счет различных прдеставленных способов.
- Оценка **уверенности** по logprobs позволяет решать, где доверять модели, а где подключать человека.
- Гибридная схема human-in-the-loop часто лучше, чем «только крауд» или «только LLM»: мы разгружаем людей на простых примерах и сохраняем качество на сложных.
- Эти идеи масштабируются дальше: улучшение промптов, дообучение модели, active learning и более умные пайплайны разметки.
